# 1. Data Loading & Cleaning

In [ ]:
# Imports neccessary libraries that this project uses
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import kagglehub


In [ ]:
path = kagglehub.dataset_download("munumbutt/superconductor-dataset")
for i in os.listdir(path):
    print(i) # Downloads and prints the csv name from kaggle

df = pd.read_csv(os.path.join(path, "train.csv")) # Load the dataset into a DataFrame (df) so we can work with it in pandas.

print(df.shape) # Prints the number of rows and columns in the data
print(df.info()) #Column names, data types and non null counts
print(df.isna().sum()) # Counts how many missing values (we dont have any)
print(df.duplicated().sum()) # counts exact duplicate rows (we have 66)
print(df.describe())  # summary stats (mean, std, min, max, quartiles) per numeric column - used to spot outliers, weird values, or scale differences
df.head() # This prints the first 5 rows of each column


In [ ]:
# Were just getting rid of those 66 duplicates now

Before = df.shape[0]

df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {Before - df.shape[0]} duplicates, {df.shape[0]} rows left")

# 2. Exploratory Data Analysis: Target Distribution

In [ ]:
tc = df["critical_temp"] #tc means "critical temperature" so we dont have to keep typing out df[xx] whenever we want it 

print(tc.describe().round(2))
print(f"Skewness: {tc.skew(): .2f}") #measures how lopsided our distribution is 

fig, ax = plt.subplots(figsize=(9, 5)) # creates the figure and the one are plot
sns.histplot(tc, bins=60, ax=ax)
ax.axvline(77, color="red", linestyle="--", label="77 K (liquid Nitrogen)")  # sets line at 77K
ax.axvline(tc.median(), color="green", linestyle="--", label=f"Median = {tc.median():.1f} K") # Line at 
ax.axvline(tc.mean(), color="black", linestyle="", label=f"mean = {tc.mean():.1f}")

ax.set_xlabel("Critical temperature, Tc (K)") #Labels x axis
ax.set_ylabel("Number of materials") #Y axis
ax.set_title("Distribution of critical temp)") 
ax.legend()
plt.show()



In [ ]:
above_77K = (tc > 77).mean() * 100
print(f"{above_77K:.1f}% of materials above 77 Kelvin with {(tc > 77).sum()} materials")


Tc ranges from near 0 K to 185 K, with a median of 20.0 K and a mean of 34.5 K. The distribution is right-skewed (skewness 0.86) and bimodal: most materials sit below 15 K, with a second cluster around 75–95 K, likely the cuprate family. 18.4% of materials (3,894) have Tc above 77 K, the boiling point of liquid nitrogen, so the classification task in section 6 is moderately imbalanced.


# 3. Exploratory Data Analysis: Feature Relationships

In [ ]:
corr = df.corr()["critical_temp"].drop("critical_temp") # Correlation of every column with Tc, removing Tc itself
corr = corr.sort_values(key=abs, ascending=False) # Sort by strength, ignoring sign (-0.6 counts as strong as 0.6)

spearman = df.corr(method="spearman")["critical_temp"].drop("critical_temp") # Rank-based version, better for skewed data

corr_table = pd.DataFrame({"pearson": corr, "spearman": spearman[corr.index]}).round(2) # Both side by side, same order
corr_table.head(20) # Show the top 20

In [ ]:
top20 = corr.head(20) # The 20 strongest correlations

plt.figure(figsize=(8, 6))
sns.barplot(x=top20.values, y=top20.index) # Horizontal bars so the long feature names fit
plt.axvline(0, color="black", linewidth=0.8) # Line at zero separates positive and negative correlations
plt.xlabel("Correlation with Tc")
plt.ylabel("") #without it, y label was saying "none"
plt.title("Top 20 features by correlation with critical temperature")
plt.tight_layout() # Stops labels getting cut off
plt.show()

In [ ]:
features = ["wtd_mean_Valence", "wtd_mean_atomic_mass", "wtd_mean_Density", "wtd_mean_ThermalConductivity", "range_ThermalConductivity", "wtd_mean_ElectronAffinity"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9)) # 6 plots, 2 rows 3 columns
for ax, feat in zip(axes.flat, features): # Loop through each plot area and feature together
    ax.scatter(df[feat], tc, s=4, alpha=0.3) # Small, see-through dots so dense areas show up
    ax.axhline(77, color="red", linestyle="--", linewidth=0.8) # 77 K line for reference
    ax.set_xlabel(feat)
    ax.set_ylabel("Tc (K)")
    ax.set_title(f"r = {df[feat].corr(tc):.2f}") # Correlation value in the title
plt.tight_layout()
plt.show()

### Scatter plots

Most features don't have a straight-line relationship with Tc. Instead, high Tc only shows up in certain ranges or clusters.

- **Valence:** high Tc only happens when valence is around 2–2.5.
- **Atomic mass:** high Tc is found at medium masses; heavy materials almost always have low Tc.
- **Density:** two separate bands reach high Tc, which suggests two different groups of materials.
- **Thermal conductivity (mean):** no clear pattern.
- **Thermal conductivity (range):** a sharp stripe at about 400 reaches high Tc. This matches materials containing copper and oxygen (cuprates), so this strong correlation is mostly caused by one type of material.
- **Electron affinity:** the correlation is weak, but high Tc only appears in a narrow range, which the correlation number misses.

Because the patterns aren't straight lines, tree-based models should do better than linear regression in Section 4.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[top20.index].corr(), cmap="RdBu_r", center =0)  #heatmap, center just makes sure 0 is always centered on the right 
plt.title("Correlation between 20 Features")
plt.show()


The top 20 features are highly inter-correlated and fall into roughly four groups: thermal conductivity spread, mean valence, compositional complexity (entropy features and number of elements), and atomic radius / ionisation energy spread. Features within each group carry largely the same information, so feature importance in section 5 will be interpreted at the level of property groups rather than individual features.

Dark red means when one goes up so does the other, blue means if one goes up the other goes down. white means if one goes up it doesnt effect the other

# 4. Regression Modelling

In [ ]:
#test

# 5. Feature Importance (Question 1)

# 6. Classification of Practical Superconductors (Question 2)

# 7. Discussion & Validation

# 8. Conclusions